Import Required Libraries

In [1]:
import pandas as pd
import sqlite3

Load Orders Data (CSV)

In [2]:
# Load transactional order data
orders_df = pd.read_csv("orders.csv")

# Convert order_date to datetime format (DD-MM-YYYY)
orders_df['order_date'] = pd.to_datetime(
    orders_df['order_date'],
    dayfirst=True
)
# Preview data
orders_df.head()

,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name
0,1,2508,450,2023-02-18,842.97,New Foods Chinese
1,2,2693,309,2023-01-18,546.68,Ruchi Curry House Multicuisine
2,3,2084,107,2023-07-15,163.93,Spice Kitchen Punjabi
3,4,319,224,2023-10-04,1155.97,Darbar Kitchen Non-Veg
4,5,1064,293,2023-12-25,1321.91,Royal Eatery South Indian


Load Users Data (JSON)

In [3]:
# Load user master data
users_df = pd.read_json("users.json")

# Preview data
users_df.head()

,user_id,name,city,membership
0,1,User_1,Chennai,Regular
1,2,User_2,Pune,Gold
2,3,User_3,Bangalore,Gold
3,4,User_4,Bangalore,Regular
4,5,User_5,Pune,Gold


Load Restaurants Data (SQL)

In [4]:
# Create in-memory SQLite database
conn = sqlite3.connect(":memory:")

# Read SQL file
with open("restaurants.sql", "r") as file:
    sql_script = file.read()

# Execute SQL script
conn.executescript(sql_script)

# Load restaurants table into DataFrame
restaurants_df = pd.read_sql(
    "SELECT * FROM restaurants",
    conn
)

# Preview data
restaurants_df.head()

,restaurant_id,restaurant_name,cuisine,rating
0,1,Restaurant_1,Chinese,4.8
1,2,Restaurant_2,Indian,4.1
2,3,Restaurant_3,Mexican,4.3
3,4,Restaurant_4,Chinese,4.1
4,5,Restaurant_5,Chinese,4.8


Standardize Column Names

In [5]:
# Convert all column names to lowercase for consistency
orders_df.columns = orders_df.columns.str.lower()
users_df.columns = users_df.columns.str.lower()
restaurants_df.columns = restaurants_df.columns.str.lower()

Merge Orders with Users (Left Join)

In [6]:
# Merge orders with users
orders_users_df = orders_df.merge(
    users_df,
    on="user_id",
    how="left"
)

orders_users_df.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name,name,city,membership
0,1,2508,450,2023-02-18,842.97,New Foods Chinese,User_2508,Hyderabad,Regular
1,2,2693,309,2023-01-18,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular
2,3,2084,107,2023-07-15,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold
3,4,319,224,2023-10-04,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold
4,5,1064,293,2023-12-25,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular


Merge with Restaurants (Left Join)

In [10]:
# Merge orders with users (Left Join to retain all orders)
orders_users_df = orders_df.merge(
    users_df,
    on="user_id",
    how="left"
)

# Merge the above with restaurants data (Left Join)
final_df = orders_users_df.merge(
    restaurants_df,
    on="restaurant_id",
    how="left"
)

final_df.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name_x,name,city,membership,restaurant_name_y,cuisine,rating
0,1,2508,450,2023-02-18,842.97,New Foods Chinese,User_2508,Hyderabad,Regular,Restaurant_450,Mexican,3.2
1,2,2693,309,2023-01-18,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular,Restaurant_309,Indian,4.5
2,3,2084,107,2023-07-15,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold,Restaurant_107,Mexican,4.0
3,4,319,224,2023-10-04,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold,Restaurant_224,Chinese,4.8
4,5,1064,293,2023-12-25,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular,Restaurant_293,Italian,3.0


Resolve Duplicate Columns

In [11]:
# Remove duplicate restaurant name coming from orders data
final_df.drop(columns=['restaurant_name_x'], inplace=True)

# Rename restaurant name from restaurant master
final_df.rename(
    columns={'restaurant_name_y': 'restaurant_name'},
    inplace=True
)

# Final column check
final_df.columns

Index(['order_id', 'user_id', 'restaurant_id', 'order_date', 'total_amount',
       'name', 'city', 'membership', 'restaurant_name', 'cuisine', 'rating'],
      dtype='object')

Feature Engineering (Time-Based Analysis)

In [12]:
# Create time-based features for trend and seasonality analysis
final_df['order_year'] = final_df['order_date'].dt.year
final_df['order_month'] = final_df['order_date'].dt.month
final_df['order_day'] = final_df['order_date'].dt.day
final_df['order_weekday'] = final_df['order_date'].dt.day_name()

Final Dataset Validation

In [13]:
# Dataset structure
final_df.info()

# Check for missing values
final_df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_id         10000 non-null  int64         
 1   user_id          10000 non-null  int64         
 2   restaurant_id    10000 non-null  int64         
 3   order_date       10000 non-null  datetime64[ns]
 4   total_amount     10000 non-null  float64       
 5   name             10000 non-null  object        
 6   city             10000 non-null  object        
 7   membership       10000 non-null  object        
 8   restaurant_name  10000 non-null  object        
 9   cuisine          10000 non-null  object        
 10  rating           10000 non-null  float64       
 11  order_year       10000 non-null  int32         
 12  order_month      10000 non-null  int32         
 13  order_day        10000 non-null  int32         
 14  order_weekday    10000 non-null  object

,0
order_id,0
user_id,0
restaurant_id,0
order_date,0
total_amount,0
name,0
city,0
membership,0
restaurant_name,0
cuisine,0


Save Final Dataset

In [14]:
# Save final dataset to CSV
final_df.to_csv(
    "final_food_delivery_dataset.csv",
    index=False
)

print("final_food_delivery_dataset.csv saved successfully")

final_food_delivery_dataset.csv saved successfully
